# NOTEBOOK UNIQUE À LANCER (révision finale)

Ce notebook fait tout seul : il télécharge les données si elles manquent, puis lance toutes les analyses.
Il peut demander **2 passages** (Kaggle limite chaque passage à 12 h).

Réglages : *Accelerator* = **None** ; *Internet* = **On**.

À la fin de chaque passage, lire la dernière ligne du journal :
* **TERMINÉ** → onglet *Output* → télécharger `resultats_revision2c.zip` et l'envoyer.
* **PAS ENCORE FINI** → *Add Input → Your Work* → choisir **ce notebook** (dernière version) → relancer *Save & Run All*.

In [ ]:
# === RÉVISION 2c : TOUT-EN-UN (cache réutilisé s'il est attaché, sinon reconstruit) ===
import base64, glob, io, os, shutil, tarfile, time, json, subprocess, sys
T0 = time.time()
def h(): return (time.time() - T0) / 3600
W = '/kaggle/working/rev2'
os.makedirs(W, exist_ok=True); os.chdir(W)
code = """..."""
tarfile.open(fileobj=io.BytesIO(base64.b64decode(code)), mode='r:gz').extractall('.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vitaldb', 'h5py'])
print('[1] code prêt')

# ---------- [2] inventaire des entrées attachées ----------
vdb_dirs, vil_dirs, done_files, excl_files, rev2_files = {}, {}, [], [], []
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    n1 = sum(1 for f in files if f.startswith('vdb') and f.endswith('.npz'))
    n2 = sum(1 for f in files if f.startswith('vil') and f.endswith('.npz'))
    if n1: vdb_dirs[root] = n1
    if n2: vil_dirs[root] = n2
    if 'done.txt' in files and n2: done_files.append(os.path.join(root, 'done.txt'))
    if 'excluded.txt' in files and n1: excl_files.append(os.path.join(root, 'excluded.txt'))
    if os.path.basename(root) == 'results_rev2': rev2_files += [os.path.join(root, f) for f in files]
print('[2] entrées : VitalDB', vdb_dirs, '| Vilnius', vil_dirs)
done_in = set()
for f in done_files: done_in |= set(open(f).read().split())
best_vdb = max(vdb_dirs.values()) if vdb_dirs else 0

# ---------- [3] cache : réutilisé directement s'il est complet, sinon fusionné et complété ----------
complete_input = best_vdb >= 400 and len(done_in) >= 39
if complete_input:
    cache = '/kaggle/working/cache_links'
    os.makedirs(cache, exist_ok=True)
    for name, dd in (('vitaldb', vdb_dirs), ('vilnius', vil_dirs)):
        dst = os.path.join(cache, name)
        if not os.path.exists(dst):
            os.makedirs(dst)
            for d in sorted(dd, key=dd.get, reverse=True):
                for f in glob.glob(d + '/*.npz'):
                    t = os.path.join(dst, os.path.basename(f))
                    if not os.path.exists(t): os.symlink(f, t)
    print('[3] cache complet trouvé dans les entrées : pas de téléchargement')
else:
    cache = 'cache'
    for sub, dd in (('vitaldb', vdb_dirs), ('vilnius', vil_dirs)):
        dst = os.path.join(cache, sub); os.makedirs(dst, exist_ok=True)
        for d in dd:
            for f in glob.glob(d + '/*.npz'):
                t = os.path.join(dst, os.path.basename(f))
                if not os.path.exists(t): shutil.copy(f, t)
    if done_in:
        old = set(open('cache/vilnius/done.txt').read().split()) if os.path.exists('cache/vilnius/done.txt') else set()
        open('cache/vilnius/done.txt', 'w').write('\n'.join(sorted(old | done_in)) + '\n')
    lines = set()
    for f in excl_files + (['cache/vitaldb/excluded.txt'] if os.path.exists('cache/vitaldb/excluded.txt') else []):
        lines |= set(l for l in open(f).read().splitlines() if l.strip())
    if lines: open('cache/vitaldb/excluded.txt', 'w').write('\n'.join(sorted(lines)) + '\n')
    print(f"[3] cache incomplet : VitalDB {len(glob.glob('cache/vitaldb/*.npz'))} cas, Vilnius {len(done_in)} patients repris")
    # VitalDB (2-3 h la première fois)
    if len(glob.glob('cache/vitaldb/*.npz')) < 400:
        subprocess.run('wget -q -r -N -c -np -nH --cut-dirs=3 -P physionet https://physionet.org/files/vitaldb-arrhythmia/1.0.0/', shell=True)
        subprocess.run(f'{sys.executable} scripts/prepare_data.py vitaldb --ann physionet/Annotation_Files --cache cache > /tmp/prep.log 2>&1', shell=True)
        subprocess.run(f'{sys.executable} scripts/relabel_vitaldb.py --ann physionet/Annotation_Files --cache cache', shell=True)
    print(f"    VitalDB : {len(glob.glob('cache/vitaldb/*.npz'))} cas  [{h():.1f} h]")
    # Vilnius (reprenable, dans la limite du temps)
    budget = max(0.0, 9.5 - h())
    if budget > 0.3:
        subprocess.run(f'{sys.executable} scripts/prepare_vilnius_stream.py --cache cache --raw /tmp/raw_vilnius --hours {budget:.2f} --threads 4', shell=True)
    done = open('cache/vilnius/done.txt').read().split() if os.path.exists('cache/vilnius/done.txt') else []
    print(f"    Vilnius : {len(done)} / 39 patients  [{h():.1f} h]")

# ---------- [4] analyses de la révision 2 ----------
os.makedirs('results_rev2', exist_ok=True)
for f in rev2_files:
    t = os.path.join('results_rev2', os.path.basename(f))
    if os.path.isfile(f) and not os.path.exists(t): shutil.copy(f, t)
done = open(os.path.join(cache, 'vilnius', 'done.txt')).read().split() if os.path.exists(os.path.join(cache, 'vilnius', 'done.txt')) else []
ready = complete_input or (len(glob.glob('cache/vitaldb/*.npz')) >= 400 and len(done) >= 39)
left = 11.3 - h()
if ready and left > 1.0:
    subprocess.run(f'{sys.executable} scripts/run_revision2b.py --cache {cache} --out results_rev2 --hours {left - 0.5:.2f}', shell=True)
    left = 11.3 - h()
    if left > 0.5:
        subprocess.run(f'{sys.executable} scripts/run_revision2c.py --cache {cache} --out results_rev2 --hours {left - 0.3:.2f}', shell=True)
subprocess.run('zip -qr /kaggle/working/resultats_revision2c.zip results_rev2', shell=True)
st = {}
for f in ('results_rev2/status_b.json', 'results_rev2/status_c.json'):
    if os.path.exists(f): st[f] = json.load(open(f))
st['finished'] = all(v.get('finished') for k, v in st.items() if k != 'finished') and len(st) == 2
print('[4]', st)
if st.get('finished'):
    print('TERMINÉ : téléchargez resultats_revision2c.zip (onglet Output) et envoyez-le.')
else:
    print('PAS ENCORE FINI : Add Input -> Your Work -> ce notebook (dernière version), gardez les autres entrées, relancez Save & Run All.')
